## Cell 1 – Setup & Reproducibility

### Mammogram BI-RADS Classification with XAI (KAU-BCMD)
### Research-grade baseline for publication

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_recall_fscore_support)
from sklearn.preprocessing import label_binarize

import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Cell 2 – Configuration

In [ ]:
class Config:
    # Paths – UPDATE THESE
    DATA_ROOT = Path("./KAU-BCMD")          # root after Kaggle download
    # Expected structure options:
    # Option A: DATA_ROOT / "BIRADS_1" / *.jpg  etc.
    # Option B: metadata CSV + image folder
    IMG_SIZE = 384                          # 224 or 384 recommended
    BATCH_SIZE = 16
    NUM_EPOCHS = 30
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 4
    NUM_CLASSES = 4                         # adjust: common is 1,3,4,5 or 1-5
    CLASS_NAMES = ["BI-RADS 1", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]  # example
    # Map original labels → 0..N-1
    LABEL_MAP = {1: 0, 3: 1, 4: 2, 5: 3}    # adjust according to your folders

cfg = Config()

## Cell 3 – Dataset Discovery & Stratified Split

In [ ]:
def discover_images(root: Path):
    """Walk the directory and collect (path, label) pairs.
    Supports both folder-per-class and CSV metadata styles.
    """
    samples = []
    # Folder-per-class style (most common on Kaggle mirrors)
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        # Try to parse BI-RADS number from folder name
        name = class_dir.name.lower()
        label = None
        for k in cfg.LABEL_MAP:
            if f"birads_{k}" in name or f"birad{k}" in name or str(k) == name[-1]:
                label = cfg.LABEL_MAP[k]
                break
        if label is None:
            continue
        for img_path in class_dir.glob("*.[jJ][pP][gG]"):
            samples.append((str(img_path), label))
        for img_path in class_dir.glob("*.[pP][nN][gG]"):
            samples.append((str(img_path), label))
    
    if not samples:
        raise RuntimeError("No images found. Check DATA_ROOT and folder naming.")
    
    df = pd.DataFrame(samples, columns=["path", "label"])
    print(df["label"].value_counts().sort_index())
    return df

df = discover_images(cfg.DATA_ROOT)

# Stratified split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, test_idx = next(sss.split(df, df["label"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

# Further split train → train/val
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(sss2.split(train_df, train_df["label"]))
train_df_final = train_df.iloc[tr_idx].reset_index(drop=True)
val_df         = train_df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_df_final)} | Val: {len(val_df)} | Test: {len(test_df)}")

## Cell 4 – Dataset Class with CLAHE + Albumentations

In [ ]:
class MammogramDataset(Dataset):
    def __init__(self, df, transform=None, is_train=False):
        self.df = df
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx]["path"]
        label = self.df.iloc[idx]["label"]
        
        # Read as grayscale then convert to 3-channel for pretrained models
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(path)
        
        # CLAHE – critical for mammograms
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = clahe.apply(img)
        
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented["image"]
        
        return img, label

# Strong medical augmentation
train_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_ds = MammogramDataset(train_df_final, train_transform, is_train=True)
val_ds   = MammogramDataset(val_df, val_transform)
test_ds  = MammogramDataset(test_df, val_transform)

# Class weights for imbalance
class_counts = train_df_final["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_df_final["label"].values]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)

## Cell 5 – Model

In [ ]:
def create_model(num_classes):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model = efficientnet_b0(weights=weights)
    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = create_model(cfg.NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(device))
optimizer = optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.NUM_EPOCHS)

## Cell 6 – Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, 100. * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return (running_loss / len(loader.dataset),
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

best_val_acc = 0.0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(cfg.NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_preds, val_labels, _ = evaluate(model, val_loader, criterion, device)
    val_acc = 100. * (val_preds == val_labels).mean()
    
    scheduler.step()
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch+1:02d} | Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
          f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_mammogram_model.pth")
        print("  → Saved best model")

print(f"Best validation accuracy: {best_val_acc:.2f}%")

## Cell 7 – Final Test Evaluation + Metrics

In [ ]:
model.load_state_dict(torch.load("best_mammogram_model.pth", map_location=device))
test_loss, test_preds, test_labels, test_probs = evaluate(model, test_loader, criterion, device)

print("\n=== Classification Report ===")
print(classification_report(test_labels, test_preds, target_names=cfg.CLASS_NAMES, digits=4))

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=cfg.CLASS_NAMES, yticklabels=cfg.CLASS_NAMES)
plt.title("Confusion Matrix – Test Set")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.show()

# One-vs-Rest AUC
y_bin = label_binarize(test_labels, classes=range(cfg.NUM_CLASSES))
aucs = roc_auc_score(y_bin, test_probs, average=None, multi_class="ovr")
print("\nPer-class AUC (OvR):")
for name, auc in zip(cfg.CLASS_NAMES, aucs):
    print(f"  {name}: {auc:.4f}")
print(f"Macro AUC: {aucs.mean():.4f}")

## Cell 8 – XAI: Grad-CAM Visualization

In [ ]:
# Target the last convolutional layer of EfficientNet-B0
target_layers = [model.features[-1]]

cam = GradCAM(model=model, target_layers=target_layers)
# Alternatives: GradCAMPlusPlus(...), HiResCAM(...)

def visualize_gradcam(dataset, indices, model, cam, class_names, device, n=6):
    model.eval()
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    
    for i, idx in enumerate(indices[:n]):
        img_tensor, label = dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        
        # Original image for overlay (denormalize)
        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np, 0, 1)
        
        targets = [ClassifierOutputTarget(label)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
        
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        
        axes[0, i].imshow(img_np)
        axes[0, i].set_title(f"True: {class_names[label]}")
        axes[0, i].axis("off")
        
        axes[1, i].imshow(cam_image)
        axes[1, i].set_title("Grad-CAM")
        axes[1, i].axis("off")
    
    plt.tight_layout()
    plt.show()

# Visualize a few test samples
sample_indices = list(range(min(12, len(test_ds))))
visualize_gradcam(test_ds, sample_indices, model, cam, cfg.CLASS_NAMES, device)

## Cell 9 – Optional: Save Predictions for Further Analysis

In [ ]:
results_df = test_df.copy()
results_df["pred"] = test_preds
results_df["prob_max"] = test_probs.max(axis=1)
for i, name in enumerate(cfg.CLASS_NAMES):
    results_df[f"prob_{name}"] = test_probs[:, i]
results_df.to_csv("test_predictions_with_xai.csv", index=False)
print("Saved test_predictions_with_xai.csv")